#Fine tuning com Optuna
Após o resultado da busca em grade, os melhores 20 modelos (10 melhores da grade com SGD e 10 melhores da grade com Adam) foram selecionados e divididdos entre os membros da equipe para execução de *fine tuning*. Dado a dificuldade em aumentar o desempenho usando métodos tradicionais de tuning, foi escolhido uma nova estratégia para prosseguir com a exploração.  
:  
O **Optuna** é uma biblioteca moderna de otimização de hiperparâmetros, utilizando algoritmos adaptativos como *Tree-structured Parzen Estimator* (TPE) para guiar a busca de maneira inteligente. Conforme novos testes são executados, o algoritmo aprende quais regiões do espaço de busca tendem a gerar melhores resultados e concentra esforços nelas, reduzindo o custo computacional e acelerando a convergência para bons modelos. Além do algoritmo adpatativo, o Optuna também utiliza o mecanismo de pruning para interromper precocemente execuções com pouco potencial, poupando tempo de execução.  
:  
O método do Optuna se baseia, resumidamente, em (1) uma função *objective*, que descreve como a exploração será conduzida; (2) um objeto *study*, que guarda os resultados da exploração.

##Importando bibliotecas

In [5]:
!pip install optuna

In [6]:
import pandas as pd
import numpy as np
import optuna

from sklearn.neural_network import MLPClassifier
from sklearn.model_selection import train_test_split
from sklearn.model_selection import StratifiedKFold
from sklearn.preprocessing import StandardScaler
from sklearn.pipeline import make_pipeline
from sklearn.metrics import f1_score
from sklearn.preprocessing import LabelEncoder

##Importando a base de dados

In [7]:
path = "ObesityDataSet_raw_and_data_sinthetic.csv"

data = pd.read_csv(path)

##Tratamento dos dados

In [8]:
#arredondando valores prejudicados pelo SMOTE
ord_disc_vars = ['Age', 'FCVC', 'NCP', 'CH2O', 'FAF', 'TUE']

for var in ord_disc_vars:
    data[var] = data[var].round().astype(int)

In [9]:
#retirando colunas com pouca relevância
low_relevance_var = ['SMOKE', 'CH2O', 'MTRANS']
data.drop(low_relevance_var, axis=1, inplace=True)

In [10]:
#one hot encoding das variáveis categóricas
categorical_vars = ['Gender', 'family_history_with_overweight', 'FAVC',
                    'FCVC', 'NCP', 'CAEC', 'SCC', 'FAF', 'TUE', 'CALC']

data = pd.get_dummies(data, columns=categorical_vars, drop_first=False)

In [11]:
#usando standard scaler nas variáveis quantitativas
quantitative_vars = ['Age', 'Height', 'Weight']

scaler = StandardScaler()
data[quantitative_vars] = scaler.fit_transform(data[quantitative_vars])

In [12]:
#colocando a variável alvo como última coluna novamente
target = data['NObeyesdad']
data = data.drop(columns=['NObeyesdad'])
data['NObeyesdad'] = target

In [13]:
#codificando coluna alvo para evitar erro no early stopping
le = LabelEncoder()
data['NObeyesdad'] = le.fit_transform(data['NObeyesdad'])

#convertendo booleanos para inteiros
for col in data.columns:
    if data[col].dtype == 'bool':
        data[col] = data[col].astype(int)


In [14]:
data.head(10)

,Age,Height,Weight,Gender_Female,Gender_Male,family_history_with_overweight_no,family_history_with_overweight_yes,FAVC_no,FAVC_yes,FCVC_1,...,FAF_2,FAF_3,TUE_0,TUE_1,TUE_2,CALC_Always,CALC_Frequently,CALC_Sometimes,CALC_no,NObeyesdad
0,-0.521741,-0.875589,-0.862558,1,0,0,1,1,0,0,...,0,0,0,1,0,0,0,0,1,1
1,-0.521741,-1.947599,-1.168077,1,0,0,1,1,0,0,...,0,1,1,0,0,0,0,1,0,1
2,-0.207057,1.054029,-0.366090,0,1,0,1,1,0,0,...,1,0,0,1,0,0,1,0,0,1
3,0.422312,1.054029,0.015808,0,1,1,0,1,0,0,...,1,0,1,0,0,0,1,0,0,5
4,-0.364399,0.839627,0.122740,0,1,1,0,1,0,0,...,0,0,1,0,0,0,0,1,0,6
5,0.736997,-0.875589,-1.282647,0,1,1,0,0,1,0,...,0,0,1,0,0,0,0,1,0,1
6,-0.207057,-2.162001,-1.206267,1,0,0,1,0,1,0,...,0,0,1,0,0,0,0,1,0,1
7,-0.364399,-0.661187,-1.282647,0,1,1,0,1,0,0,...,0,1,1,0,0,0,0,1,0,1
8,-0.049714,0.839627,-0.862558,0,1,0,1,0,1,0,...,0,0,0,1,0,0,1,0,0,1
9,-0.364399,0.196421,-0.709799,0,1,0,1,0,1,0,...,0,0,0,1,0,0,0,0,1,1


In [15]:
# Preparando os dados
coluna_alvo = 'NObeyesdad'
X = (data.drop(columns=[coluna_alvo])).values
Y = (data[coluna_alvo]).values

##Tuning dos modelos

In [16]:
#preset dos modelos
top5 = [
    {"hidden_layer_sizes": (4, 6), "activation": "relu", "batch_size": 32, "learning_rate_init": 0.001, "solver": "adam", "max_iter": 300, "iter_no_change": 50},
    {"hidden_layer_sizes": (9,),   "activation": "tanh", "batch_size": 128, "learning_rate_init": 0.01,  "solver": "adam", "max_iter": 200, "iter_no_change": 50},
    {"hidden_layer_sizes": (5,),   "activation": "relu", "batch_size": 32, "learning_rate_init": 0.01,  "solver": "adam", "max_iter": 200, "iter_no_change": 30},
    {"hidden_layer_sizes": (4, 6), "activation": "relu", "batch_size": 20, "learning_rate_init": 0.001, "solver": "adam", "max_iter": 200, "iter_no_change": 20},
    {"hidden_layer_sizes": (6, 4), "activation": "relu", "batch_size": 100,"learning_rate_init": 0.01,  "solver": "adam", "max_iter": 300, "iter_no_change": 30}
]

In [36]:
#configurações do estudo
num_trials = 200
direction = "maximize"

In [37]:
#função objetivo
def objective(trial, X, y):
  '''
  Função objetivo do Optuna.
  '''

  #escolhe um preset aleatório
  preset = trial.suggest_categorical("preset_idx", list(range(len(top5))))
  base = top5[preset]

  #parâmetros fixos
  solver = base["solver"]

  #oscilando numero de camadas
  num_layers = trial.suggest_int("num_hidden_layers", 1, 2)

  #oscilando numero de neuronios
  first_layer = trial.suggest_int("h1", 3, 11)
  if num_layers == 2:
      second_layer = trial.suggest_int("h2", 3, 11)
      hidden_layer_sizes = (first_layer, second_layer)
  else:
      hidden_layer_sizes = (first_layer,)

  #oscilação da função de ativação
  activation = trial.suggest_categorical(
      "activation",
       ["relu", "tanh", "logistic"]
      )

  #oscilação da taxa de learning rate
  lr = trial.suggest_float(
      "learning_rate_init",
      base["learning_rate_init"] / 5,
      base["learning_rate_init"] * 5,
      log=True
      )

  #oscilação do batch size
  batch = trial.suggest_categorical(
      f"batch_size_{preset}",
       [max(8, base["batch_size"] // 2),
        base["batch_size"],
        min(256, base["batch_size"] * 2)]
      )


  #oscilação do max iter
  max_iter = int(
      trial.suggest_int(
          "max_iter",
          max(150, base["max_iter"] - 100),
          base["max_iter"] + 300
          )
      )

  #oscilação do iter_no_change
  iter_no_change = trial.suggest_int(
      "iter_no_change",
      max(5, base["iter_no_change"] - 30),
      base["iter_no_change"] + 30
      )

  #montando modelo
  clf = MLPClassifier(
        hidden_layer_sizes=hidden_layer_sizes,
        activation=activation,
        solver=solver,
        learning_rate_init=lr,
        batch_size=batch,
        max_iter=max_iter,
        early_stopping=True,
        n_iter_no_change=iter_no_change,
        random_state=42
        )
  pipe = make_pipeline(StandardScaler(), clf)

  #cross validation
  skf = StratifiedKFold(n_splits=5, shuffle=True, random_state=42)
  scores = []

  for fold, (train_idx, val_idx) in enumerate(skf.split(X, y)):
      X_train, X_val = X[train_idx], X[val_idx]
      y_train, y_val = y[train_idx], y[val_idx]

      pipe.fit(X_train, y_train)
      pred = pipe.predict(X_val)
      scores.append(f1_score(y_val, pred, average="macro"))

      trial.report(np.mean(scores), fold)

      if trial.should_prune():
          raise optuna.TrialPruned()

  return np.mean(scores)

In [38]:
#para gerar log
optuna.logging.set_verbosity(optuna.logging.INFO)

#inicia o estudo
study = optuna.create_study(direction=direction)
study.optimize(lambda trial: objective(trial, X, Y), n_trials=num_trials)

#mostra os resultados
print("\nMelhor score:", study.best_value)
print("Melhores parâmetros:", study.best_params)

[I 2025-11-29 23:30:41,165] A new study created in memory with name: no-name-f2d7b6fb-5b72-42c9-bd54-7a755b4683b8
[I 2025-11-29 23:30:45,876] Trial 0 finished with value: 0.9063865524091803 and parameters: {'preset_idx': 1, 'num_hidden_layers': 1, 'h1': 9, 'activation': 'relu', 'learning_rate_init': 0.0066986646382351306, 'batch_size_1': 128, 'max_iter': 354, 'iter_no_change': 29}. Best is trial 0 with value: 0.9063865524091803.
/usr/local/lib/python3.12/dist-packages/sklearn/neural_network/_multilayer_perceptron.py:691: ConvergenceWarning: Stochastic Optimizer: Maximum iterations (246) reached and the optimization hasn't converged yet.
  warnings.warn(
[I 2025-11-29 23:30:52,818] Trial 1 finished with value: 0.8938726586209385 and parameters: {'preset_idx': 4, 'num_hidden_layers': 2, 'h1': 5, 'h2': 7, 'activation': 'tanh', 'learning_rate_init': 0.040002872220484244, 'batch_size_4': 100, 'max_iter': 246, 'iter_no_change': 60}. Best is trial 0 with value: 0.9063865524091803.
[I 2025-11-


Melhor score: 0.9605425454162548
Melhores parâmetros: {'preset_idx': 3, 'num_hidden_layers': 2, 'h1': 5, 'h2': 11, 'activation': 'logistic', 'learning_rate_init': 0.0017534025796977614, 'batch_size_3': 10, 'max_iter': 447, 'iter_no_change': 40}


In [39]:
#salva o log
df = study.trials_dataframe()
df.to_csv("optuna_trials.csv", index=False)

##Discussão

Durante o processo de *tuning*, a função objective foi alterada algumas vezes, mudando o alcance de oscilação das alterações, quais seriam os hiperparâmetros fixos, entre outros, com o objetivo de promover a exploração e melhorar o desempenho. Mesmo com diversas iterações e com o máximo de parâmetros sendo explorados, o melhor modelo que o Optuna conseguiu achar alcança somente 0.9605 de F1 Macro, ainda longe do modelo TOP 1 encontrado pela grade (0.9739).  
```
Melhor score: 0.9605
Melhores parâmetros: {'hidden_layers': (5, 11),
                      'activation': 'logistic',
                      'learning_rate_init': 0.00175,
                      'batch_size': 10,
                      'max_iter': 447,
                      'iter_no_change': 40}
```
Considerando as 200 iterações e a maximização de parâmetros sendo tunados, e que mesmo diante dessa exploração não foi possível achar um modelo com desempenho igual ou melhor, é seguro dizer que o resultado da grade já oferece resultados muito bem ajustados ao problema proposto.